# 05b — GeniZ: Metapath Selection Experiment

**Purpose**: Test a leaner metapath configuration compared to `05_GeniZ` (the validated baseline).

**Hypothesis**: Several metapaths in the final selection show negative PFI drops, indicating redundancy. A more parsimonious set may achieve comparable or better performance while being more interpretable.

**Baseline results (05_GeniZ — DO NOT OVERWRITE)**:
- Metapaths: MP14, MP2, MP10, MP5, MP11 (Rama A) + MP4, MP7, MP8, MP12, MP13 (Rama B) = 10 total
- Spearman=0.6639 | NDCG=0.4376 | Combined=0.5508 | RBP95=0.0780

**Experiment**: Cap at 4 metapaths per branch, ensuring MP4 is always included in Rama B.

**Strategy**:
- Rama A: top-4 by forward selection (drop MP11 — negative NDCG drop in PFI)
- Rama B: MP4 forced + top-3 by Lite attention (drop lowest contributor)

**Outputs saved to**: `GeniZ/reports_exp/` and `GeniZ/models_exp/` — isolated from baseline.

## 1. Setup — identical to 05_GeniZ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pickle, glob, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.metrics import r2_score, ndcg_score
from sklearn.mixture import GaussianMixture
from scipy.stats import spearmanr
from tqdm import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | PyTorch: {torch.__version__}')

In [ ]:
BASE_DIR   = '/content/drive/MyDrive/metapath_analysis'  # Update to your Google Drive path
GENIZ_DIR  = os.path.join(BASE_DIR, 'GeniZ')
EMB_DIR    = os.path.join(BASE_DIR, 'metapath2vec_embeddings')

PAGERANK_PATH = os.path.join(GENIZ_DIR, 'pagerank_scores_zscore.pkl')
E2I_PATH      = os.path.join(GENIZ_DIR, 'entity2idx_zscore.pkl')

# Isolated output dirs — never touch reports_v2 / models_v2
MODEL_DIR   = os.path.join(GENIZ_DIR, 'models_exp')
REPORTS_DIR = os.path.join(GENIZ_DIR, 'reports_exp')
os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Hyperparameters — identical to baseline
EMB_DIM    = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT    = 0.3
BATCH_SIZE = 512
LR           = 1.5e-4
WEIGHT_DECAY = 5e-5
EPOCHS       = 40
FREEZE_CENT  = 8
WARMUP       = 5
THR_LOW  = 0.114722
W_LOW    = 6.0
THR_HIGH = 0.610942
W_HIGH   = 16.0
BPR_MAX                 = 0.075
BPR_STEP                = 0.015
BPR_HOLD_EPOCHS         = 4
BPR_COOLDOWN_PATIENCE   = 4
BPR_ACTIVATION_PATIENCE = 5
ATTN_STABLE_TOL         = 0.015
ATTN_STABLE_WINDOW      = 3
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

print('Configuration set')

In [ ]:
SCHEMA_MAP = {
    'metapath_1_emb' : 'Firm - FoS - Firm',
    'metapath_2_emb' : 'Firm - FoS - FoS - Firm',
    'metapath_3_emb' : 'FoS - Firm - FoS',
    'metapath_4_emb' : 'Firm - Patent - Country - Patent - Firm',
    'metapath_5_emb' : 'Firm - Product - FoS - Product - Firm',
    'metapath_6_emb' : 'Product - FoS - Product',
    'metapath_7_emb' : 'Firm - Country - Firm',
    'metapath_8_emb' : 'Firm - Univ - Country - Univ - Firm',
    'metapath_9_emb' : 'Patent - Country - Patent',
    'metapath_10_emb': 'FoS - Product - FoS',
    'metapath_11_emb': 'Product - FoS - Firm - FoS - Product',
    'metapath_12_emb': 'Product - Firm - Product',
    'metapath_13_emb': 'Univ - Firm - Univ',
    'metapath_14_emb': 'FoS - FoS - FoS',
}
def mp_label(tag):
    return tag.replace('_emb','').replace('metapath_','MP')

print('Schema map ready')

## 2. Load Data

In [ ]:
with open(PAGERANK_PATH, 'rb') as f:
    pagerank_zscore = pickle.load(f)
with open(E2I_PATH, 'rb') as f:
    entity2idx = pickle.load(f)
entity2idx_rev = {v: k for k, v in entity2idx.items()}

print(f'PageRank z-scores: {len(pagerank_zscore):,} nodes')
print(f'entity2idx       : {len(entity2idx):,} nodes')

In [ ]:
def load_embeddings(emb_dir, allowed=None):
    embeddings, node_maps = {}, {}
    for fname in sorted(os.listdir(emb_dir)):
        if not fname.endswith('.pkl'): continue
        tag = fname.replace('.pkl', '')
        if allowed and tag not in allowed: continue
        with open(os.path.join(emb_dir, fname), 'rb') as f:
            data = pickle.load(f)
        emb = torch.tensor(data['embeddings'], dtype=torch.float32)
        id2node = data['id2node']
        node2row = {v: k for k, v in (
            id2node.items() if isinstance(id2node, dict) else enumerate(id2node))}
        embeddings[tag] = emb
        node_maps[tag]  = node2row
    print(f'Loaded {len(embeddings)} embeddings')
    return embeddings, node_maps

emb_all, map_all = load_embeddings(EMB_DIR)
print('Embeddings ready')

## 3. Experimental Metapath Sets

### Rationale for trimming

From PFI in 05_GeniZ (baseline with 10 metapaths):

| Metapath | NDCG drop | Interpretation |
|---|---|---|
| MP4  | +92.24% | Critical — backbone of institutional dimension |
| MP2  | +0.04%  | Marginal for top-k; useful for global ranking |
| MP8  | +1.21%  | Small but positive |
| MP11 | -0.54%  | **Redundant** — removing improves NDCG |
| MP13 | -2.72%  | **Redundant** — removing improves NDCG |
| MP12 | -0.21%  | Marginal negative |
| MP7  | -0.26%  | Marginal negative |

**Decision**:
- Rama A: keep top-4 from forward selection (drop MP11 — redundant)
- Rama B: force MP4 + keep top-3 by Lite attention (drop MP13 — redundant, and MP12 — marginal)

Three variants tested to isolate the effect of each trimming decision.

In [ ]:
# Baseline (from 05_GeniZ) — reference only, not re-trained here
BASELINE = {
    'name'    : 'Baseline (05_GeniZ)',
    'rama_a'  : ['metapath_14_emb','metapath_2_emb','metapath_10_emb',
                 'metapath_5_emb','metapath_11_emb'],
    'rama_b'  : ['metapath_4_emb','metapath_7_emb','metapath_8_emb',
                 'metapath_12_emb','metapath_13_emb'],
    'metrics' : {'spearman':0.6639,'ndcg':0.4376,'combined':0.5508,
                 'rbp_95':0.0780,'rbp_80':0.1770,'r2':0.1275},
}

# Experiment A — drop MP11 (redundant in Rama A), keep Rama B intact
EXP_A = {
    'name'  : 'Exp-A: Trim Rama A (drop MP11)',
    'rama_a': ['metapath_14_emb','metapath_2_emb',
               'metapath_10_emb','metapath_5_emb'],
    'rama_b': ['metapath_4_emb','metapath_7_emb','metapath_8_emb',
               'metapath_12_emb','metapath_13_emb'],
}

# Experiment B — drop MP13 and MP12 from Rama B (redundant), keep Rama A intact
EXP_B = {
    'name'  : 'Exp-B: Trim Rama B (drop MP13, MP12)',
    'rama_a': ['metapath_14_emb','metapath_2_emb','metapath_10_emb',
               'metapath_5_emb','metapath_11_emb'],
    'rama_b': ['metapath_4_emb','metapath_7_emb','metapath_8_emb'],
}

# Experiment C — trim both branches (most parsimonious)
EXP_C = {
    'name'  : 'Exp-C: Trim both (4+3 = 7 metapaths)',
    'rama_a': ['metapath_14_emb','metapath_2_emb',
               'metapath_10_emb','metapath_5_emb'],
    'rama_b': ['metapath_4_emb','metapath_7_emb','metapath_8_emb'],
}

EXPERIMENTS = [EXP_A, EXP_B, EXP_C]

for exp in EXPERIMENTS:
    mp_all = exp['rama_a'] + exp['rama_b']
    exp['mp_keys'] = mp_all
    print(f"{exp['name']}")
    print(f"  Rama A ({len(exp['rama_a'])}): {[mp_label(m) for m in exp['rama_a']]}")
    print(f"  Rama B ({len(exp['rama_b'])}): {[mp_label(m) for m in exp['rama_b']]}")
    print(f"  Total: {len(mp_all)} metapaths\n")

## 4. Model, Dataset, and Training Utilities

Identical architecture to 05_GeniZ — no changes.

In [ ]:
class GENIDataset(Dataset):
    def __init__(self, pagerank, entity2idx, embeddings, node_maps, allowed=None):
        self.entity2idx = entity2idx
        self.embeddings = embeddings
        self.node_maps  = node_maps
        self.mp_keys    = allowed or sorted(embeddings.keys())
        self.emb_dim    = next(iter(embeddings.values())).shape[1]
        self.node_ids   = [nid for nid in entity2idx if nid in pagerank]
        self.targets    = {nid: pagerank[nid] for nid in self.node_ids}

    def __len__(self): return len(self.node_ids)

    def __getitem__(self, idx):
        nid  = self.node_ids[idx]
        nidx = self.entity2idx[nid]
        y    = self.targets[nid]
        emb_dict = {}
        for tag in self.mp_keys:
            row = self.node_maps[tag].get(nid)
            emb_dict[tag] = (self.embeddings[tag][row] if row is not None
                             else torch.zeros(self.emb_dim))
        return emb_dict, nidx, float(y), 1.0


def collate_geni(batch):
    mp_keys  = list(batch[0][0].keys())
    emb_b    = {k: torch.stack([b[0][k] for b in batch]) for k in mp_keys}
    node_ids = torch.tensor([b[1] for b in batch], dtype=torch.long)
    y_b      = torch.tensor([b[2] for b in batch], dtype=torch.float32)
    w_b      = torch.tensor([b[3] for b in batch], dtype=torch.float32)
    return emb_b, node_ids, y_b, w_b


def make_splits(dataset, seed=42):
    N   = len(dataset)
    idx = np.random.default_rng(seed).permutation(N)
    n_tr, n_va = int(N*TRAIN_FRAC), int(N*VAL_FRAC)
    return (Subset(dataset, idx[:n_tr]),
            Subset(dataset, idx[n_tr:n_tr+n_va]),
            Subset(dataset, idx[n_tr+n_va:]))


def make_loaders(dataset, train_s, val_s, test_s):
    zs = np.array([dataset.targets[dataset.node_ids[i]] for i in train_s.indices])
    w  = np.ones_like(zs, dtype=np.float32)
    w[zs > THR_HIGH] = W_HIGH
    w[(zs > THR_LOW) & (zs <= THR_HIGH)] = W_LOW
    sampler = WeightedRandomSampler(torch.tensor(w), len(w))
    tr = DataLoader(train_s, BATCH_SIZE, sampler=sampler, collate_fn=collate_geni)
    va = DataLoader(val_s,   BATCH_SIZE, shuffle=False,   collate_fn=collate_geni)
    te = DataLoader(test_s,  BATCH_SIZE, shuffle=False,   collate_fn=collate_geni)
    return tr, va, te


class GeniModel(nn.Module):
    def __init__(self, metapath_names, num_nodes,
                 embedding_dim=128, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.metapath_names = metapath_names
        self.metapath_mlps  = nn.ModuleDict({
            name: nn.ModuleList(
                [nn.Sequential(
                    nn.Linear(embedding_dim if i==0 else hidden_dim, hidden_dim),
                    nn.ReLU(), nn.Dropout(dropout)
                ) for i in range(num_layers)]
                + [nn.Linear(hidden_dim, 1)]
            ) for name in metapath_names
        })
        self.attn_logits       = nn.Parameter(torch.randn(len(metapath_names)))
        self.centrality_params = nn.Parameter(torch.zeros(num_nodes, 1))
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)

    def get_attention_weights(self):
        with torch.no_grad():
            w = torch.softmax(self.attn_logits.cpu(), dim=0).numpy()
        return {name: float(w[i]) for i, name in enumerate(self.metapath_names)}

    def forward(self, emb_dict, node_ids):
        scores, masks = [], []
        for name in self.metapath_names:
            x     = F.normalize(emb_dict[name], p=2, dim=1)
            valid = (x.abs().sum(dim=1) > 1e-6).float()
            for layer in self.metapath_mlps[name][:-1]: x = layer(x)
            scores.append(self.metapath_mlps[name][-1](x).squeeze(-1))
            masks.append(valid)
        S = torch.stack(scores, dim=1)
        M = torch.stack(masks,  dim=1)
        logits = self.attn_logits.unsqueeze(0).expand_as(S)
        logits = logits * M + (1 - M) * (-1e9)
        A      = F.softmax(logits, dim=1)
        y_hat  = (A * S).sum(dim=1)
        y_hat  = y_hat + self.centrality_params[node_ids.long()].squeeze(1)
        return y_hat


def band_w(z):
    w = torch.ones_like(z)
    w = torch.where(z > THR_HIGH, torch.as_tensor(W_HIGH, device=z.device, dtype=z.dtype), w)
    w = torch.where((z > THR_LOW) & (z <= THR_HIGH),
                    torch.as_tensor(W_LOW, device=z.device, dtype=z.dtype), w)
    return w


def rbp_score(yt, yp, p=0.95):
    order = np.argsort(yp)[::-1]
    ys    = yt[order]
    mn, mx = ys.min(), ys.max()
    rel  = (ys - mn) / (mx - mn) if mx > mn else np.ones_like(ys)
    return float((1-p) * np.sum(rel * (p ** np.arange(len(rel)))))


def evaluate(model, loader, perturb_mp=None):
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for emb_dict, node_ids, y, _ in loader:
            if perturb_mp and perturb_mp in emb_dict:
                emb_dict[perturb_mp] = emb_dict[perturb_mp][
                    torch.randperm(emb_dict[perturb_mp].size(0))]
            node_ids = node_ids.to(device).long()
            for k in emb_dict: emb_dict[k] = emb_dict[k].to(device)
            yp.extend(model(emb_dict, node_ids).cpu().numpy())
            yt.extend(y.numpy())
    yt, yp = np.array(yt), np.array(yp)
    ytn    = yt - yt.min()
    spr    = float(spearmanr(yt, yp).statistic)
    nd     = float(ndcg_score(ytn.reshape(1,-1), yp.reshape(1,-1), k=100))
    return {'spearman': spr, 'ndcg': nd, 'combined': (spr+nd)/2.0,
            'r2': float(r2_score(yt, yp)),
            'rbp_95': rbp_score(yt, yp, 0.95),
            'rbp_80': rbp_score(yt, yp, 0.80)}


def run_training(model, tr_l, va_l, epochs, name):
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    lf    = nn.HuberLoss(reduction='none', delta=1.0)
    ck_nd   = os.path.join(MODEL_DIR, f'{name}_best_ndcg.pt')
    ck_comb = os.path.join(MODEL_DIR, f'{name}_best_combined.pt')
    best_nd = -1.0; best_comb = -1.0; no_imp = 0
    bpr_on = False; bpr_c = 0.0; bpr_hold = 0
    attn_prev = None; attn_hist = []

    def get_attn():
        with torch.no_grad():
            return torch.softmax(model.attn_logits.cpu(), dim=0).numpy()

    def is_stable(d):
        attn_hist.append(d)
        if len(attn_hist) < ATTN_STABLE_WINDOW: return False
        return all(x <= ATTN_STABLE_TOL for x in attn_hist[-ATTN_STABLE_WINDOW:])

    for epoch in range(1, epochs + 1):
        lr = LR * epoch / WARMUP if epoch <= WARMUP else LR
        for pg in opt.param_groups: pg['lr'] = lr
        model.centrality_params.requires_grad_(epoch > FREEZE_CENT)

        model.train(); tr_loss = 0.0
        for emb_dict, node_ids, y, _ in tqdm(tr_l, leave=False):
            node_ids = node_ids.to(device).long()
            y = y.to(device)
            for k in emb_dict: emb_dict[k] = emb_dict[k].to(device)
            y_hat = model(emb_dict, node_ids)
            raw   = lf(y_hat, y)
            l1    = 5e-6 * model.attn_logits.abs().sum()
            l2    = 1e-5 * model.centrality_params.pow(2).sum()
            loss  = (raw * band_w(y)).mean() + l1 + l2
            if bpr_on and bpr_c > 0:
                pos = y > THR_HIGH; neg = y <= THR_LOW
                if pos.any() and neg.any():
                    yp2, yn2 = y_hat[pos], y_hat[neg]
                    k_p = min(512, yp2.shape[0]*yn2.shape[0])
                    ip  = torch.randint(0, yp2.shape[0], (k_p,), device=device)
                    ig  = torch.randint(0, yn2.shape[0], (k_p,), device=device)
                    loss = loss + bpr_c * (-torch.log(
                        torch.sigmoid(yp2[ip]-yn2[ig])+1e-8).mean())
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step(); tr_loss += loss.item()
        tr_loss /= max(1, len(tr_l))

        m  = evaluate(model, va_l)
        av = get_attn()
        delta = float(np.abs(av - attn_prev).sum()) if attn_prev is not None else float('nan')
        attn_prev = av.copy()
        ds = f'{delta:.4f}' if np.isfinite(delta) else 'nan'

        print(f'Ep {epoch:02d} | loss={tr_loss:.4f} | '
              f'spr={m["spearman"]:.4f} | ndcg={m["ndcg"]:.4f} | '
              f'comb={m["combined"]:.4f} | rbp95={m["rbp_95"]:.4f} | '
              f'BPR={"ON" if bpr_on else "OFF"}({bpr_c:.3f}) | da={ds}')
        sched.step(tr_loss)

        if m['ndcg'] > best_nd + 1e-6:
            best_nd = m['ndcg']; no_imp = 0
            torch.save(model.state_dict(), ck_nd)
            print(f'  Best NDCG -> {best_nd:.4f}')
        else:
            no_imp += 1
        if m['combined'] > best_comb:
            best_comb = m['combined']
            torch.save(model.state_dict(), ck_comb)
            print(f'  Best Comb -> {best_comb:.4f}')

        stable = is_stable(delta if np.isfinite(delta) else 1e9)
        if not bpr_on and no_imp >= BPR_ACTIVATION_PATIENCE and stable:
            bpr_on = True; bpr_c = BPR_STEP; bpr_hold = 1
            print(f'  [BPR] Activated coeff={bpr_c:.3f}')
        elif bpr_on:
            if bpr_hold < BPR_HOLD_EPOCHS: bpr_hold += 1
            elif m['ndcg'] > best_nd - 1e-6 and bpr_c < BPR_MAX:
                bpr_c = min(bpr_c + BPR_STEP, BPR_MAX)
                print(f'  [BPR] Ramp up coeff={bpr_c:.3f}')
            elif no_imp >= BPR_COOLDOWN_PATIENCE:
                bpr_c = max(0.0, bpr_c - BPR_STEP)
                if bpr_c == 0.0: bpr_on = False; print('  [BPR] Deactivated')

    print(f'Done. Best NDCG={best_nd:.4f} | Best Combined={best_comb:.4f}')
    return ck_nd, ck_comb


print('All utilities defined')

## 5. Run Experiments

Each experiment trains a GeniZ Final model on its metapath set and evaluates on the same test split (SEED=42). Results are collected in `comparison_table` for direct comparison with the baseline.

In [ ]:
comparison = []

# Add baseline as reference row
comparison.append({
    'name'     : BASELINE['name'],
    'n_mp'     : len(BASELINE['rama_a']) + len(BASELINE['rama_b']),
    'rama_a'   : [mp_label(m) for m in BASELINE['rama_a']],
    'rama_b'   : [mp_label(m) for m in BASELINE['rama_b']],
    **BASELINE['metrics']
})

print('Baseline loaded as reference.')
print('Starting experiments...\n')

In [ ]:
# Run all 3 experiments sequentially
for exp in EXPERIMENTS:
    print(f'\n{"="*60}')
    print(f'  {exp["name"]}')
    print(f'  Rama A: {[mp_label(m) for m in exp["rama_a"]]}')
    print(f'  Rama B: {[mp_label(m) for m in exp["rama_b"]]}')
    print(f'{"="*60}\n')

    mp_keys = exp['mp_keys']
    emb_exp = {k: emb_all[k] for k in mp_keys}
    map_exp = {k: map_all[k] for k in mp_keys}

    ds  = GENIDataset(pagerank_zscore, entity2idx, emb_exp, map_exp, mp_keys)
    tr, va, te = make_splits(ds, SEED)
    tr_l, va_l, te_l = make_loaders(ds, tr, va, te)

    model = GeniModel(
        metapath_names=mp_keys, num_nodes=len(entity2idx),
        embedding_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)

    safe_name = exp['name'].replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')
    ck_nd, ck_comb = run_training(model, tr_l, va_l, EPOCHS, safe_name)

    # Evaluate on test set with best combined checkpoint
    model.load_state_dict(torch.load(ck_comb, map_location=device))
    test_m = evaluate(model, te_l)

    # PFI
    baseline_test = evaluate(model, te_l)
    pfi = {}
    for mp in mp_keys:
        m_p = evaluate(model, te_l, perturb_mp=mp)
        pfi[mp_label(mp)] = round(
            (baseline_test['ndcg'] - m_p['ndcg']) / abs(baseline_test['ndcg']) * 100, 2)

    attn = model.get_attention_weights()

    row = {
        'name'   : exp['name'],
        'n_mp'   : len(mp_keys),
        'rama_a' : [mp_label(m) for m in exp['rama_a']],
        'rama_b' : [mp_label(m) for m in exp['rama_b']],
        **test_m,
        'pfi'    : pfi,
        'attn'   : {mp_label(k): round(v,4) for k,v in attn.items()},
    }
    comparison.append(row)
    exp['results'] = row

    print(f'\n  TEST RESULTS:')
    print(f'  Spearman={test_m["spearman"]:.4f} | NDCG={test_m["ndcg"]:.4f} | '
          f'Combined={test_m["combined"]:.4f} | RBP95={test_m["rbp_95"]:.4f}')
    print(f'  PFI: {pfi}')

print('\nAll experiments done.')

## 6. Comparison Table

In [ ]:
rows = []
for r in comparison:
    rows.append({
        'Model'    : r['name'],
        'N mp'     : r['n_mp'],
        'Spearman' : round(r['spearman'], 4),
        'NDCG@100' : round(r['ndcg'], 4),
        'Combined' : round(r['combined'], 4),
        'RBP(0.95)': round(r['rbp_95'], 4),
        'RBP(0.80)': round(r['rbp_80'], 4),
        'R2'       : round(r['r2'], 4),
    })

df_comp = pd.DataFrame(rows)
print('=== Comparison Table ===')
print(df_comp.to_string(index=False))

df_comp.to_csv(os.path.join(REPORTS_DIR, 'comparison_table.csv'), index=False)
print('\nSaved: comparison_table.csv')

# Highlight best per metric
print('\n=== Best per metric ===')
for col in ['Spearman','NDCG@100','Combined','RBP(0.95)']:
    best_idx = df_comp[col].idxmax()
    print(f'  {col:12}: {df_comp.loc[best_idx,"Model"]} ({df_comp.loc[best_idx,col]:.4f})')

In [ ]:
# Visual comparison
metrics = ['Spearman','NDCG@100','Combined','RBP(0.95)']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

colours = ['#2c3e50', '#e74c3c', '#3498db', '#2ecc71']
models  = df_comp['Model'].tolist()
labels  = ['Baseline\n(10 mp)', 'Exp-A\n(9 mp)', 'Exp-B\n(8 mp)', 'Exp-C\n(7 mp)']

for ax, metric in zip(axes, metrics):
    vals = df_comp[metric].tolist()
    bars = ax.bar(labels, vals, color=colours, alpha=0.85)
    ax.set_title(metric, fontsize=11)
    ax.set_ylim(0, max(vals)*1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    # Baseline reference line
    ax.axhline(vals[0], color='#2c3e50', linestyle='--', linewidth=0.8, alpha=0.5)

plt.suptitle('GeniZ — Metapath Trimming Experiment', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_chart.png')

## 7. Conclusion

**Decision criteria**:
- If Exp-C (7 metapaths) achieves Combined within 0.02 of baseline → prefer Exp-C (parsimony)
- If NDCG drop > 0.03 in any experiment → keep baseline
- MP4 must be present in all selected configurations (PFI confirmed it as structural backbone)

Update this cell with the final decision after reviewing results.

In [ ]:
def to_python(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f'Not serializable: {type(obj)}')

summary_exp = {
    'baseline'   : BASELINE,
    'experiments': [
        {k: v for k, v in r.items() if k != 'pfi'}
        for r in comparison[1:]
    ],
    'pfi_by_exp' : [
        {'name': r['name'], 'pfi': r.get('pfi', {})}
        for r in comparison[1:]
    ]
}

with open(os.path.join(REPORTS_DIR, 'experiment_summary.json'), 'w') as f:
    json.dump(summary_exp, f, indent=2, default=to_python)

print('All saved in:', REPORTS_DIR)
print('  - comparison_table.csv')
print('  - comparison_chart.png')
print('  - experiment_summary.json')
print('  - models_exp/*.pt')

In [ ]:
# ── Reconstruir variables perdidas en la desconexión ─────────────────────
import os, pickle
import torch
import numpy as np

SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_DIR   = '/content/drive/MyDrive/metapath_analysis'  # Update to your Google Drive path
GENIZ_DIR  = os.path.join(BASE_DIR, 'GeniZ')
EMB_DIR    = os.path.join(BASE_DIR, 'metapath2vec_embeddings')
MODEL_DIR  = os.path.join(GENIZ_DIR, 'models_exp')
REPORTS_DIR= os.path.join(GENIZ_DIR, 'reports_exp')

SCHEMA_MAP = {
    'metapath_1_emb' : 'Firm - FoS - Firm',
    'metapath_2_emb' : 'Firm - FoS - FoS - Firm',
    'metapath_3_emb' : 'FoS - Firm - FoS',
    'metapath_4_emb' : 'Firm - Patent - Country - Patent - Firm',
    'metapath_5_emb' : 'Firm - Product - FoS - Product - Firm',
    'metapath_6_emb' : 'Product - FoS - Product',
    'metapath_7_emb' : 'Firm - Country - Firm',
    'metapath_8_emb' : 'Firm - Univ - Country - Univ - Firm',
    'metapath_9_emb' : 'Patent - Country - Patent',
    'metapath_10_emb': 'FoS - Product - FoS',
    'metapath_11_emb': 'Product - FoS - Firm - FoS - Product',
    'metapath_12_emb': 'Product - Firm - Product',
    'metapath_13_emb': 'Univ - Firm - Univ',
    'metapath_14_emb': 'FoS - FoS - FoS',
}
def mp_label(tag):
    return tag.replace('_emb','').replace('metapath_','MP')

EXP_C = {
    'name'  : 'Exp-C: Trim both (4+3 = 7 metapaths)',
    'rama_a': ['metapath_14_emb','metapath_2_emb',
               'metapath_10_emb','metapath_5_emb'],
    'rama_b': ['metapath_4_emb','metapath_7_emb','metapath_8_emb'],
}
EXP_C['mp_keys'] = EXP_C['rama_a'] + EXP_C['rama_b']

with open(os.path.join(GENIZ_DIR, 'pagerank_scores_zscore.pkl'), 'rb') as f:
    pagerank_zscore = pickle.load(f)
with open(os.path.join(GENIZ_DIR, 'entity2idx_zscore.pkl'), 'rb') as f:
    entity2idx = pickle.load(f)
entity2idx_rev = {v: k for k, v in entity2idx.items()}

def load_embeddings(emb_dir, allowed=None):
    embeddings, node_maps = {}, {}
    for fname in sorted(os.listdir(emb_dir)):
        if not fname.endswith('.pkl'): continue
        tag = fname.replace('.pkl', '')
        if allowed and tag not in allowed: continue
        with open(os.path.join(emb_dir, fname), 'rb') as f:
            data = pickle.load(f)
        emb = torch.tensor(data['embeddings'], dtype=torch.float32)
        id2node = data['id2node']
        node2row = {v: k for k, v in (
            id2node.items() if isinstance(id2node, dict) else enumerate(id2node))}
        embeddings[tag] = emb
        node_maps[tag]  = node2row
    return embeddings, node_maps

emb_all, map_all = load_embeddings(EMB_DIR, allowed=EXP_C['mp_keys'])

print(f'Device: {device}')
print(f'Embeddings cargados: {list(emb_all.keys())}')
print(f'EXP_C mp_keys: {[mp_label(m) for m in EXP_C["mp_keys"]]}')
print('Variables restauradas — puedes correr las celdas del GeniZ Final')

In [ ]:
# ── GeniZ Final — Exp-C configuration (4+3 = 7 metapaths) ────────────────
print('=== GeniZ Final — Exp-C (MP14, MP2, MP10, MP5 | MP4, MP7, MP8) ===\n')

MP_KEYS_FINAL = EXP_C['mp_keys']

emb_fin = {k: emb_all[k] for k in MP_KEYS_FINAL}
map_fin = {k: map_all[k] for k in MP_KEYS_FINAL}

ds_fin = GENIDataset(pagerank_zscore, entity2idx, emb_fin, map_fin, MP_KEYS_FINAL)
tr_fin, va_fin, te_fin = make_splits(ds_fin, SEED)
tr_l_fin, va_l_fin, te_l_fin = make_loaders(ds_fin, tr_fin, va_fin, te_fin)

model_fin = GeniModel(
    metapath_names=MP_KEYS_FINAL, num_nodes=len(entity2idx),
    embedding_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)

print(f'Parameters: {sum(p.numel() for p in model_fin.parameters()):,}')
ck_fin_nd, ck_fin_comb = run_training(
    model_fin, tr_l_fin, va_l_fin, EPOCHS, 'GeniZ_Final_ExpC')

In [ ]:
# ── Test evaluation + PFI + attention weights ─────────────────────────────
model_fin.load_state_dict(torch.load(ck_fin_comb, map_location=device))
test_m = evaluate(model_fin, te_l_fin)

print('=== Test Set Evaluation — GeniZ Final Exp-C ===')
print(f'  Spearman rho : {test_m["spearman"]:.4f}')
print(f'  NDCG@100     : {test_m["ndcg"]:.4f}')
print(f'  Combined     : {test_m["combined"]:.4f}')
print(f'  RBP (p=0.95) : {test_m["rbp_95"]:.4f}')
print(f'  RBP (p=0.80) : {test_m["rbp_80"]:.4f}')
print(f'  R2           : {test_m["r2"]:.4f}')

# Attention weights
attn_fin = model_fin.get_attention_weights()
print('\n=== Attention Weights ===')
for tag, w in sorted(attn_fin.items(), key=lambda x: -x[1]):
    branch = 'A' if tag in EXP_C['rama_a'] else 'B'
    print(f'  [{branch}] {mp_label(tag):>5}: {w:.4f}  {SCHEMA_MAP[tag]}')

# PFI
print('\n=== Permutation Feature Importance ===')
baseline = evaluate(model_fin, te_l_fin)
pfi_rows = []
for mp in MP_KEYS_FINAL:
    m = evaluate(model_fin, te_l_fin, perturb_mp=mp)
    drop_nd  = (baseline['ndcg']     - m['ndcg'])     / abs(baseline['ndcg'])     * 100
    drop_spr = (baseline['spearman'] - m['spearman']) / abs(baseline['spearman']) * 100
    drop_rbp = (baseline['rbp_95']   - m['rbp_95'])   / abs(baseline['rbp_95'])   * 100
    pfi_rows.append({
        'Metapath'         : mp_label(mp),
        'Schema'           : SCHEMA_MAP[mp],
        'Branch'           : 'A' if mp in EXP_C['rama_a'] else 'B',
        'NDCG drop (%)'    : round(drop_nd, 2),
        'Spearman drop (%)': round(drop_spr, 2),
        'RBP drop (%)'     : round(drop_rbp, 2),
    })
    print(f'  {mp_label(mp):>5}: NDCG drop={drop_nd:.2f}% | Spr drop={drop_spr:.2f}%')

df_pfi_fin = pd.DataFrame(pfi_rows).sort_values('NDCG drop (%)', ascending=False)
print('\n=== Ranked by NDCG drop ===')
print(df_pfi_fin.to_string(index=False))

# Save
df_pfi_fin.to_csv(os.path.join(REPORTS_DIR, 'pfi_final_expc.csv'), index=False)
pd.DataFrame([test_m]).to_csv(os.path.join(REPORTS_DIR, 'test_metrics_final_expc.csv'), index=False)

def to_python(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f'Not serializable: {type(obj)}')

summary_final = {
    'model'      : 'GeniZ Final Exp-C',
    'mp_keys'    : MP_KEYS_FINAL,
    'rama_a'     : EXP_C['rama_a'],
    'rama_b'     : EXP_C['rama_b'],
    'test_metrics': test_m,
    'attn_weights': {mp_label(k): round(v,4) for k,v in attn_fin.items()},
    'pfi'        : df_pfi_fin.to_dict('records'),
    'vs_baseline': {
        'spearman_delta': round(test_m['spearman'] - 0.6639, 4),
        'ndcg_delta'    : round(test_m['ndcg']     - 0.4376, 4),
        'combined_delta': round(test_m['combined'] - 0.5508, 4),
        'rbp95_delta'   : round(test_m['rbp_95']   - 0.0780, 4),
    }
}
with open(os.path.join(REPORTS_DIR, 'final_expc_summary.json'), 'w') as f:
    json.dump(summary_final, f, indent=2, default=to_python)

print('\n=== vs Baseline (05_GeniZ) ===')
for k, v in summary_final['vs_baseline'].items():
    sign = '+' if v >= 0 else ''
    print(f'  {k}: {sign}{v:.4f}')

print('\nAll saved to reports_exp/')

In [ ]:
# ── Core-Periphery GMM — GeniZ Final Exp-C ───────────────────────────────
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

# Predicciones sobre todos los nodos
model_fin.eval()
all_nidx, all_preds, all_true = [], [], []

full_loader = DataLoader(ds_fin, BATCH_SIZE, shuffle=False, collate_fn=collate_geni)

with torch.no_grad():
    for emb_dict, node_ids, y, _ in tqdm(full_loader, desc='Predicting'):
        node_ids = node_ids.to(device).long()
        for k in emb_dict: emb_dict[k] = emb_dict[k].to(device)
        all_preds.extend(model_fin(emb_dict, node_ids).cpu().numpy())
        all_nidx.extend(node_ids.cpu().numpy())
        all_true.extend(y.numpy())

df_rank = pd.DataFrame({
    'node_idx'     : all_nidx,
    'global_id'    : [entity2idx_rev[i] for i in all_nidx],
    'pagerank_true': all_true,
    'score_pred'   : all_preds,
})
df_rank['prefix']    = df_rank['global_id'].str.extract(r'^([a-zA-Z]+)')
df_rank['rank_true'] = df_rank['pagerank_true'].rank(ascending=False).astype(int)
df_rank['rank_pred'] = df_rank['score_pred'].rank(ascending=False).astype(int)

print(f'Predictions: {len(df_rank):,} nodes')
print(df_rank['prefix'].value_counts())

In [ ]:
# ── GMM con BIC ───────────────────────────────────────────────────────────
df_firms = df_rank[df_rank['prefix'] == 'firm'].copy()
scores   = df_firms['score_pred'].values.reshape(-1, 1)

bic = {}
for n in [2, 3, 4]:
    g = GaussianMixture(n_components=n, random_state=SEED, n_init=5)
    g.fit(scores)
    bic[n] = g.bic(scores)
    print(f'  GMM n={n}: BIC={bic[n]:.1f}')

best_n = min(bic, key=bic.get)
print(f'Best n_components = {best_n}')

gmm    = GaussianMixture(n_components=best_n, random_state=SEED, n_init=10)
gmm.fit(scores)
labels = gmm.predict(scores)
means  = gmm.means_.flatten()
order  = np.argsort(means)[::-1]

zone_names_all = ['core', 'semi-core', 'semi-periphery', 'periphery']
zones  = zone_names_all[:best_n]
c2z    = {comp: zones[rank] for rank, comp in enumerate(order)}

df_firms = df_firms.copy()
df_firms['zone'] = [c2z[c] for c in labels]

print('\n=== Core-Periphery Distribution (firms) — Exp-C ===')
for zone in zones:
    n   = (df_firms['zone'] == zone).sum()
    pct = n / len(df_firms) * 100
    mpr = df_firms[df_firms['zone'] == zone]['pagerank_true'].mean()
    print(f'  {zone:<14}: {n:>7,}  ({pct:5.1f}%)  mean PR z-score={mpr:.4f}')

df_firms.to_csv(os.path.join(REPORTS_DIR, 'core_periphery_expc.csv'), index=False)
print('\nSaved: core_periphery_expc.csv')

# ── Plot ──────────────────────────────────────────────────────────────────
colours = {'core': '#e74c3c', 'semi-core': '#e67e22',
           'semi-periphery': '#f1c40f', 'periphery': '#3498db'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1 — distribución completa log scale
for zone in zones:
    sub = df_firms[df_firms['zone'] == zone]['score_pred']
    axes[0].hist(sub, bins=80, alpha=0.6, label=zone, color=colours[zone])
axes[0].set_yscale('log')
axes[0].set_xlabel('GeniZ predicted score')
axes[0].set_ylabel('Firm count (log)')
axes[0].set_title('Full distribution (log scale)')
axes[0].legend(fontsize=8)

# Panel 2 — zoom no-core (score < 5)
for zone in [z for z in zones if z != 'core']:
    sub = df_firms[(df_firms['zone'] == zone) & (df_firms['score_pred'] < 5)]['score_pred']
    axes[1].hist(sub, bins=60, alpha=0.6, label=zone, color=colours[zone])
axes[1].set_xlabel('GeniZ predicted score')
axes[1].set_ylabel('Firm count')
axes[1].set_title('Non-core firms (score < 5)')
axes[1].legend(fontsize=8)

# Panel 3 — tamaños
sizes = [int((df_firms['zone'] == z).sum()) for z in zones]
axes[2].bar(zones, sizes, color=[colours[z] for z in zones])
axes[2].set_ylabel('Number of firms')
axes[2].set_title('Zone sizes')
for i, v in enumerate(sizes):
    axes[2].text(i, v + 200, f'{v:,}', ha='center', fontsize=8)

plt.suptitle('Core-Periphery Structure — GeniZ Final Exp-C (7 metapaths)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'core_periphery_expc.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: core_periphery_expc.png')

In [ ]:
# ── Summary final — GeniZ Final Exp-C ────────────────────────────────────
from datetime import datetime
import json
import numpy as np

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M')

def to_python(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    raise TypeError(f'Not serializable: {type(obj)}')

summary_final = {
    'run_id'          : RUN_ID,
    'model'           : 'GeniZ Final Exp-C',
    'mp_keys_final'   : MP_KEYS_FINAL,
    'rama_a'          : EXP_C['rama_a'],
    'rama_b'          : EXP_C['rama_b'],
    'test_metrics'    : test_m,
    'attn_weights'    : {mp_label(k): round(v,4) for k,v in attn_fin.items()},
    'pfi'             : df_pfi_fin.to_dict('records'),
    'core_periphery'  : {
        zone: {
            'n'             : int((df_firms['zone']==zone).sum()),
            'pct'           : round(float((df_firms['zone']==zone).sum()/len(df_firms)*100), 2),
            'mean_pr_zscore': round(float(df_firms[df_firms['zone']==zone]['pagerank_true'].mean()), 4),
        }
        for zone in zones
    },
    'gmm_n_components': int(best_n),
    'vs_baseline'     : {
        'spearman_delta': round(test_m['spearman'] - 0.6639, 4),
        'ndcg_delta'    : round(test_m['ndcg']     - 0.4376, 4),
        'combined_delta': round(test_m['combined'] - 0.5508, 4),
        'rbp95_delta'   : round(test_m['rbp_95']   - 0.0780, 4),
        'n_metapaths_saved': 3,
    },
}

path = os.path.join(REPORTS_DIR, f'summary_final_expc_{RUN_ID}.json')
with open(path, 'w', encoding='utf-8') as f:
    json.dump(summary_final, f, indent=2, ensure_ascii=False, default=to_python)

# ── Imprimir resumen ejecutivo ────────────────────────────────────────────
print('=' * 55)
print('  GENI Z FINAL EXP-C — RESUMEN EJECUTIVO')
print('=' * 55)
print(f'\n  Metapaths ({len(MP_KEYS_FINAL)}):')
for mp in MP_KEYS_FINAL:
    branch = 'A' if mp in EXP_C['rama_a'] else 'B'
    print(f'    [{branch}] {mp_label(mp):>5} — {SCHEMA_MAP[mp]}')

print(f'\n  Métricas test set:')
print(f'    Spearman rho : {test_m["spearman"]:.4f}')
print(f'    NDCG@100     : {test_m["ndcg"]:.4f}')
print(f'    Combined     : {test_m["combined"]:.4f}')
print(f'    RBP (p=0.95) : {test_m["rbp_95"]:.4f}')
print(f'    RBP (p=0.80) : {test_m["rbp_80"]:.4f}')
print(f'    R2           : {test_m["r2"]:.4f}')

print(f'\n  vs Baseline (10 metapaths):')
for k, v in summary_final['vs_baseline'].items():
    if k == 'n_metapaths_saved':
        print(f'    Metapaths eliminados : {v}')
    else:
        sign = '+' if v >= 0 else ''
        print(f'    {k:20}: {sign}{v:.4f}')

print(f'\n  Core-Periphery (GMM, n={best_n}):')
for zone, vals in summary_final['core_periphery'].items():
    print(f'    {zone:<14}: {vals["n"]:>7,}  ({vals["pct"]:5.1f}%)  '
          f'z={vals["mean_pr_zscore"]:.4f}')

print(f'\n  Archivos guardados en: {REPORTS_DIR}')
print(f'    summary_final_expc_{RUN_ID}.json')
print(f'    test_metrics_final_expc.csv')
print(f'    pfi_final_expc.csv')
print(f'    core_periphery_expc.csv')
print(f'    core_periphery_expc.png')
print(f'    comparison_table.csv')
print(f'    comparison_chart.png')
print('=' * 55)